# W2-D2 — Phân tích nguyên nhân gốc (Graph + Causal + LLM-augmented)

Ta lấy **output của correlator D1** (`results/cluster_summary.json`) — các cluster alert
liên quan — và với mỗi cluster, quyết định **service nào là thủ phạm (culprit)** so với
service nào chỉ là nạn nhân của cascade. Kết hợp ba tín hiệu:

1. **Graph traversal** — PageRank trên subgraph *phụ thuộc* dịch vụ (service được phụ
   thuộc nhiều nhất là gốc khả dĩ).
2. **Temporal** — service alert *sớm nhất* được cộng điểm.
3. **LLM-augmented** — Claude phân loại sự cố và đề xuất hành động, dựa trên các incident
   tương tự đã truy hồi (RAG).

Toàn bộ logic nằm trong `rca.py`; notebook này điều khiển nó và ghi `results/rca_output.json`.

In [1]:
import json, inspect
import networkx as nx
import rca

cluster_summary = rca.load_json("results/cluster_summary.json")
alerts          = rca.load_jsonl("lab/dataset/alerts_sample.jsonl")
services_doc    = rca.load_json("lab/dataset/services.json")
history         = rca.load_json("lab/dataset/incidents_history.json")

print("clusters :", cluster_summary["n_clusters"])
print("alerts   :", len(alerts))
print("services :", len(services_doc["services"]))
print("incidents:", len(history))
main = cluster_summary["clusters"][0]
print("\nMain cluster", main["cluster_id"], "services:", main["services"])

clusters : 3
alerts   : 80
services : 11
incidents: 30

Main cluster c-001-000 services: ['edge-lb', 'checkout-svc', 'payment-svc', 'notification-svc', 'cart-svc', 'inventory-svc']


## 1. Dựng service dependency graph

Cạnh `A -> B` nghĩa là **A gọi / phụ thuộc B**. Lỗi lan *ngược* chiều cạnh: nếu `B` hỏng,
mọi caller của `B` cũng hỏng theo. Nên root cause thường nằm ở **đáy** dependency graph
(sink được phụ thuộc nhiều nhất).

In [2]:
graph = rca.build_graph(services_doc)
print("nodes:", graph.number_of_nodes(), " edges:", graph.number_of_edges())

sub = graph.subgraph(main["services"])
print("\nDependencies inside the main cluster (caller -> callee):")
for u, v in sub.edges():
    print(f"  {u} -> {v}")
print("\npayment-svc is called by:", sorted(graph.predecessors("payment-svc")))

nodes: 11  edges: 14

Dependencies inside the main cluster (caller -> callee):
  edge-lb -> checkout-svc
  edge-lb -> cart-svc
  checkout-svc -> payment-svc
  checkout-svc -> inventory-svc
  checkout-svc -> cart-svc
  checkout-svc -> notification-svc
  cart-svc -> payment-svc
  notification-svc -> payment-svc

payment-svc is called by: ['cart-svc', 'checkout-svc', 'notification-svc']


## 2. Graph + temporal RCA

`rca_pagerank` chạy PageRank trên subgraph caller→callee, nên rank tích lũy ở service được
phụ thuộc nhiều nhất. `rca_combined` trộn PageRank đã chuẩn hóa (0.6) với điểm temporal
theo alert sớm nhất (0.4).

In [3]:
pr = rca.rca_pagerank(main["services"], graph)
print("Raw PageRank (main cluster):")
for k, v in sorted(pr.items(), key=lambda x: -x[1]):
    print(f"  {k:18s} {v:.4f}")
print("confidence (top/sum): %.3f" % (max(pr.values()) / sum(pr.values())))

scoped = rca.cluster_alerts(main, alerts)
print("\nEarliest alert per service:")
for s, t in sorted(rca._earliest_per_service(main["services"], scoped).items(), key=lambda x: x[1]):
    print(f"  {s:18s} {t}")

print("\nCombined graph+temporal ranking (top candidates):")
for svc, sc in rca.rca_combined(main, alerts, graph):
    print(f"  {svc:18s} {sc:.4f}")

Raw PageRank (main cluster):
  payment-svc        0.3646
  cart-svc           0.1624
  checkout-svc       0.1340
  inventory-svc      0.1225
  notification-svc   0.1225
  edge-lb            0.0940
confidence (top/sum): 0.365

Earliest alert per service:
  payment-svc        2026-06-08T03:14:05Z
  checkout-svc       2026-06-08T03:14:40Z
  cart-svc           2026-06-08T03:15:10Z
  inventory-svc      2026-06-08T03:15:30Z
  notification-svc   2026-06-08T03:16:00Z
  edge-lb            2026-06-08T03:16:30Z

Combined graph+temporal ranking (top candidates):
  payment-svc        1.0000
  checkout-svc       0.5404
  cart-svc           0.5073
  inventory-svc      0.3615
  notification-svc   0.2815
  edge-lb            0.1547


## 3. Truy hồi — incident quá khứ tương tự (RAG)

`incidents_history.json` chứa 30 incident quá khứ. `top_k_similar` xếp hạng chúng so với
cluster hiện tại theo service gốc trùng nhau, service overlap, và severity khớp — các case
truy hồi được đưa vào LLM làm context nền.

In [4]:
for it in rca.top_k_similar(main, history, k=3):
    print(f"{it['id']}  sim={it['_similarity']}  "
          f"root_cause={it['root_cause_service']} ({it['root_cause_class']})")
    print("   ", it["summary"])

INC-2026-05-30  sim=1.0  root_cause=payment-svc (connection_pool_exhaustion)
    payment-svc v3.2 shipped with pool size 50; Black-Friday traffic exhausted it, checkout cascaded.
INC-2026-05-24  sim=1.0  root_cause=payment-svc (connection_pool_exhaustion)
    payment-svc DB pool acquire timeouts after a slow downstream provider held connections open.
INC-2026-05-18  sim=0.8  root_cause=payment-svc (slow_query)
    Missing index on payments.txn(merchant_id, created_at) made settlement query table-scan.


## 4. RCA tăng cường bằng LLM (Claude, output theo JSON-schema)

`rca.call_llm_rca` gọi **Anthropic Messages API** kèm JSON-schema (`output_config.format`)
để đảm bảo phản hồi luôn parse được. Model: `claude-opus-4-8` (không có `temperature` — đã
bị bỏ trên Opus 4.8). Khi không có `ANTHROPIC_API_KEY`, pipeline fallback về heuristic
deterministic dựa trên truy hồi nên vẫn cho output hợp lệ khi chạy offline.

In [5]:
print("LLM available (ANTHROPIC_API_KEY + sdk):", rca.llm_available())
print("model:", rca.LLM_MODEL)
print("\nThe Claude call (from rca.call_llm_rca):")
src = inspect.getsource(rca.call_llm_rca)
print("\n".join(l for l in src.splitlines() if "client.messages.create" in l
                  or "output_config" in l or "model=" in l or "anthropic.Anthropic" in l))

result_main = rca.run_rca(main, alerts, graph, history)
print("\nRCA result for", main["cluster_id"], ":")
print(json.dumps(result_main, indent=2))

LLM available (ANTHROPIC_API_KEY + sdk): False
model: claude-opus-4-8

The Claude call (from rca.call_llm_rca):
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
    # output_config.format guarantees the first text block is schema-valid JSON.
    resp = client.messages.create(
        model=LLM_MODEL,
        output_config={"format": {"type": "json_schema", "schema": RCA_SCHEMA}},

RCA result for c-001-000 :
{
  "cluster_id": "c-001-000",
  "graph_top3": [
    [
      "payment-svc",
      1.0
    ],
    [
      "checkout-svc",
      0.5404
    ],
    [
      "cart-svc",
      0.5073
    ]
  ],
  "root_cause": "payment-svc",
  "class": "connection_pool_exhaustion",
  "confidence": 0.9,
  "actions": [
    "Rolled back payment-svc to v3.1 and raised HikariCP pool 50 -> 120."
  ],
  "reasoning": "payment-svc is the most-depended-on service in the cluster and alerts earliest, so the cascade originates there. Past incident INC-2026-05-30 (connection_pool_exhaustion) on p

## 5. Chạy full pipeline và ghi `results/rca_output.json`

In [6]:
doc = rca.run_all(
    cluster_summary_path="results/cluster_summary.json",
    alerts_path="lab/dataset/alerts_sample.jsonl",
    services_path="lab/dataset/services.json",
    history_path="lab/dataset/incidents_history.json",
    out_path="results/rca_output.json",
)
print("clusters_analyzed:", doc["clusters_analyzed"], " llm_used:", doc["llm_used"])
for r in doc["results"]:
    print(f"  {r['cluster_id']}: {r['root_cause']:16s} ({r['class']}) "
          f"conf={r['confidence']}  method={r['method']}")

clusters_analyzed: 3  llm_used: False
  c-001-000: payment-svc      (connection_pool_exhaustion) conf=0.9  method=graph+retrieval-heuristic
  c-002-000: cart-redis       (connection_pool_exhaustion) conf=0.9  method=graph+retrieval-heuristic
  c-003-000: kafka-broker     (rebalance_storm) conf=0.9  method=graph+retrieval-heuristic


## 6. Kiểm tra acceptance

In [7]:
doc = rca.load_json("results/rca_output.json")
assert doc["clusters_analyzed"] == 3
for r in doc["results"]:
    assert r["graph_top3"] and r["root_cause"] and r["class"]
    ok, errs = rca.validate_llm_output(r, next(c for c in cluster_summary["clusters"]
                                               if c["cluster_id"] == r["cluster_id"]))
    assert ok, (r["cluster_id"], errs)
assert doc["results"][0]["root_cause"] == "payment-svc"
assert doc["results"][1]["store_diagnostic"]["verdict"] == "store_is_culprit"
print("ALL CHECKS PASSED — rca_output.json is valid with graph_top3 + root_cause + class.")

ALL CHECKS PASSED — rca_output.json is valid with graph_top3 + root_cause + class.
